# Sinh mô tả chi tiết cho 5000 ảnh bằng Qwen2.5-VL-7B (vLLM, GPU L4)

Cùng câu lệnh với bản thử 20 ảnh (v2): tả **từng người** trên ảnh cắt theo khung YOLO, **3 vùng** phóng to (đảo bếp +
bồn rửa, bếp nấu, phòng khách), và **cả ảnh** (đếm người, vị trí). Bản thử đo được ~3.3 giây/ảnh → **~4–5 giờ**.

**Chuẩn bị (1 lần, trên máy):** tải lên Google Drive, thư mục `MyDrive/image_retrieval_colab/`:
- `anh_5000.zip` — 5000 ảnh (nằm trong thư mục `colab/` của repo, ~1.2 GB)
- `khung_nguoi_5000.json` — khung người YOLO (cũng trong `colab/`)

**Cách chạy:**
1. Runtime → Change runtime type → **L4**. Chạy ô 1, ô 2, rồi **Runtime → Restart session** (bắt buộc).
2. Bấm vào ô **2b** → **Runtime → Run cell and below**, rồi để đó.

**Chạy theo từng phần, có thể làm tiếp:** mỗi phần 200 ảnh được lưu ngay lên Drive (`mo_ta_5000/part_XXX.jsonl`).
Nếu Colab ngắt giữa chừng (hết giờ, mất mạng, lỗi), chỉ cần chạy lại như trên: phần đã xong được **bỏ qua**, làm tiếp
từ phần còn thiếu.

**Tự ngắt phiên:** khi xong, khi lỗi, hoặc quá 7 giờ — lưu nhật ký vào `nhat_ky.txt` rồi ngắt để không tốn compute unit.
Kết quả cuối: `MyDrive/image_retrieval_colab/mo_ta_5000.jsonl` → tải về và gửi lại.

In [ ]:
# 1. Kiểm tra GPU (cần L4 hoặc A100)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Cài đặt (~3–5 phút). GHIM PHIÊN BẢN: vLLM mới nhất (0.34) lỗi khi khởi động vì thiếu gói "cutlass"
#    (module của model MiniMax, không liên quan Qwen) -> dùng bản ổn định đã hỗ trợ Qwen2.5-VL + transformers khớp.
!pip install -q "vllm==0.10.2" "transformers==4.56.2" qwen-vl-utils
import importlib.metadata as _md
print({p: _md.version(p) for p in ("vllm", "transformers", "torch")})
# BẮT BUỘC: Runtime -> Restart session (phiên bản torch/transformers vừa đổi), rồi bấm ô 2b -> Run cell and below.

In [ ]:
# 2b. BẢO VỆ PHIÊN: lưu kết quả vào Google Drive và TỰ NGẮT phiên GPU khi xong / khi lỗi / khi quá giờ.
import os, time, shutil, threading, traceback
from google.colab import drive, runtime
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/image_retrieval_colab"
os.makedirs(OUT, exist_ok=True)
KET_QUA_FILES = ["toc_do.txt"]   # từng phần đã lưu thẳng lên Drive
GIOI_HAN_GIO = 7          # 5000 ảnh cần ~4–5 giờ; quá 7 giờ coi như bị treo -> ngắt (chạy lại sẽ làm tiếp)
TU_NGAT = True            # đặt False nếu muốn giữ phiên để xem / chạy lại bằng tay
_da_ngat = False

def ghi_nhat_ky(dong):
    with open(f"{OUT}/nhat_ky.txt", "a", encoding="utf-8") as f:
        f.write(f"{time.strftime('%Y-%m-%d %H:%M:%S')} {dong}\n")

def luu_ket_qua():
    for fn in KET_QUA_FILES:
        if os.path.exists(fn):
            shutil.copy(fn, OUT)

def ngat(ly_do):
    global _da_ngat
    if _da_ngat:
        return
    _da_ngat = True
    try:
        ghi_nhat_ky(f"NGẮT PHIÊN: {ly_do}")
        luu_ket_qua()
        drive.flush_and_unmount()          # đảm bảo file đã ghi xong lên Drive
        print(f"Đã lưu vào {OUT}. Ngắt phiên: {ly_do}")
    finally:
        if TU_NGAT:
            runtime.unassign()             # trả GPU, dừng tính compute unit

def _khi_loi(shell, etype, evalue, tb, tb_offset=None):
    shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)
    ghi_nhat_ky("LỖI:\n" + "".join(traceback.format_exception(etype, evalue, tb))[-4000:])
    ngat(f"lỗi {etype.__name__}: {evalue}")

# Chỉ bắt Exception: bấm Stop (KeyboardInterrupt) thì không ngắt phiên.
get_ipython().set_custom_exc((Exception,), _khi_loi)
_hen_gio = threading.Timer(GIOI_HAN_GIO * 3600, lambda: ngat(f"quá {GIOI_HAN_GIO} giờ (có thể bị treo)"))
_hen_gio.daemon = True
_hen_gio.start()
ghi_nhat_ky("BẮT ĐẦU")
print(f"Tự ngắt: bật (xong / lỗi / quá {GIOI_HAN_GIO} giờ). Kết quả lưu ở {OUT}")

In [ ]:
# 3. Chép ảnh từ Drive về ổ máy Colab (đọc nhanh hơn đọc thẳng từ Drive) + đọc khung người
import os, json, zipfile, glob
ZIP = f"{OUT}/anh_5000.zip"
KHUNG = f"{OUT}/khung_nguoi_5000.json"
for f in (ZIP, KHUNG):
    if not os.path.exists(f):
        raise FileNotFoundError(f"Chưa có {f} trên Drive — xem phần Chuẩn bị ở đầu notebook")
if not os.path.exists("anh_5000/.xong"):
    print("Đang chép và giải nén ~1.2 GB...")
    shutil.copy(ZIP, "/content/anh_5000.zip")
    zipfile.ZipFile("/content/anh_5000.zip").extractall("anh_5000")
    open("anh_5000/.xong", "w").close()
duong_dan = {os.path.basename(p): p for p in glob.glob("anh_5000/**/*.jpg", recursive=True)}
khung_all = json.load(open(KHUNG))
anh = sorted(f for f in khung_all if f in duong_dan)
thieu = [f for f in khung_all if f not in duong_dan]
print(len(anh), "ảnh sẵn sàng", f"({len(thieu)} ảnh trong danh sách không tìm thấy file)" if thieu else "")
ghi_nhat_ky(f"{len(anh)} ảnh, thiếu {len(thieu)}")

In [ ]:
# 4. Nạp model. Thử vLLM trước (nhanh); lỗi thì tự chuyển sang transformers (chậm hơn, cùng model).
#    GPU T4 (16 GB, không có bfloat16): đổi MODEL sang "Qwen/Qwen2.5-VL-3B-Instruct" và DTYPE = "float16".
#    Báo hết bộ nhớ GPU sau khi vLLM lỗi: kết nối lại, đặt DUNG_VLLM = False, chạy lại từ ô 2b.
MODEL = "Qwen/Qwen2.5-VL-7B-Instruct"
DTYPE = "bfloat16"
DUNG_VLLM = True
MAX_PIXELS = 1280 * 720
llm = None
if DUNG_VLLM:
    try:
        from vllm import LLM, SamplingParams
        llm = LLM(model=MODEL, dtype=DTYPE, max_model_len=8192, gpu_memory_utilization=0.90,
                  limit_mm_per_prompt={"image": 1},
                  mm_processor_kwargs={"min_pixels": 64 * 28 * 28, "max_pixels": MAX_PIXELS})
        import vllm
        print("Đang dùng vLLM", vllm.__version__)
    except Exception as e:
        print("vLLM lỗi, chuyển sang transformers (CHẬM ~30 giây/ảnh):", repr(e)[:300])
        ghi_nhat_ky("vLLM lỗi: " + repr(e)[:1500])
        llm = None
if llm is None:
    import gc, torch
    gc.collect(); torch.cuda.empty_cache()
    from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
    hf_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL, torch_dtype=getattr(torch, DTYPE), device_map="cuda", attn_implementation="sdpa").eval()
    hf_proc = AutoProcessor.from_pretrained(MODEL, min_pixels=64 * 28 * 28, max_pixels=MAX_PIXELS)
    hf_proc.tokenizer.padding_side = "left"
    print("Đang dùng transformers")

In [ ]:
# 5. Câu lệnh — KHÔNG có ví dụ cụ thể (v1 bị model chép ví dụ). Chỉ khung trống + bắt ghi "none".
NGUOI = """This image is a crop of ONE person from an overhead fisheye kitchen camera (the person may look small or
distorted; parts of other people may appear at the edges - ignore them). Describe only the main person, only what is
clearly visible. If something is not visible or you are not sure, write "none". Do not guess.
Replace every <...> placeholder with your own words; never output the angle brackets themselves.
Return valid JSON only:
{"gender": "...", "hair": "...", "top": "<colour> <garment type> <pattern if any>", "bottom": "...",
 "headwear": "...", "eyewear": "...", "other_accessories": "...",
 "holding": "<object(s) in the hands with colour, or none>", "action": "...",
 "posture": "standing | sitting | bending | walking", "facing": "toward camera | back to camera | sideways"}"""

VUNG = {
    "island_sink": ("the kitchen island with its sink and faucet", (0.20, 0.20, 0.66, 0.66)),
    "stove":       ("the stove area", (0.50, 0.08, 0.80, 0.58)),
    "living_room": ("the living room with the sofa", (0.03, 0.08, 0.36, 0.50)),
}
def vung_prompt(ten):
    return f"""This image is a zoomed crop of {ten} from an overhead kitchen camera. List only what you can clearly
see. Do not assume typical kitchen items. If nothing is there, use an empty list.
List each object at most once and at most 12 objects. Replace the <...> placeholders with your own words; never output
the angle brackets themselves.
Return valid JSON only:
{{"objects": ["<colour> <object> <where exactly, e.g. on/next to/hanging from what>", ...],
 "people": ["<clothing, headwear, hair, posture, action>", ...]}}"""

CA_ANH = """This is a full frame from a fixed overhead fisheye camera in a home kitchen with a living room in the back.
Count every visible person, including small or partly hidden people in the back. Describe only what is visible; do not
guess. Return valid JSON only:
{"people_count": <integer>,
 "people": [{"where_in_room": "...", "where_in_frame": "<e.g. bottom-left, center, top-right>", "short": "<clothing + action>"}],
 "caption": "<2-4 sentences mentioning every person and any small coloured objects you can clearly see>"}"""

In [ ]:
# 6. Chạy theo từng phần 200 ảnh. Phần nào đã có trên Drive thì bỏ qua (chạy lại sau khi bị ngắt sẽ làm tiếp).
import base64, io, re, time
from PIL import Image
def cat_anh(im, box, pad=0.0, min_side=224):
    W, H = im.size
    x1, y1, x2, y2 = box
    w, h = x2 - x1, y2 - y1
    x1, x2 = max(0, x1 - pad * w), min(1, x2 + pad * w)
    y1, y2 = max(0, y1 - pad * h), min(1, y2 + pad * h)
    c = im.crop((int(x1 * W), int(y1 * H), int(x2 * W), int(y2 * H)))
    s = min_side / max(1, min(c.size))
    if s > 1:                                    # phóng to người nhỏ để model nhìn rõ
        c = c.resize((int(c.width * s), int(c.height * s)), Image.BICUBIC)
    return c


def b64(img):
    buf = io.BytesIO(); img.save(buf, format="JPEG", quality=92)
    return base64.b64encode(buf.getvalue()).decode()

def doc_json(txt):
    m = re.search(r"\{.*\}", txt, re.S)
    try:
        return json.loads(m.group(0)) if m else None
    except json.JSONDecodeError:
        return None

def chay_transformers(ds, lo=6):
    import torch
    kq = []
    for i in range(0, len(ds), lo):
        batch = ds[i:i + lo]
        texts = [hf_proc.apply_chat_template([{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": p}]}],
                                             tokenize=False, add_generation_prompt=True) for _, p in batch]
        inp = hf_proc(text=texts, images=[img for img, _ in batch], return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            g = hf_model.generate(**inp, max_new_tokens=700, do_sample=False, repetition_penalty=1.05)
        kq += hf_proc.batch_decode(g[:, inp["input_ids"].shape[1]:], skip_special_tokens=True)
    return kq

if llm is None:     # transformers ~34 giây/ảnh -> ~47 giờ cho 5000 ảnh: dừng (tự ngắt phiên) thay vì đốt compute unit
    raise RuntimeError("vLLM không khởi động được (xem nhat_ky.txt) — không chạy 5000 ảnh bằng transformers vì quá chậm")
TOI_DA = {"nguoi": 300, "vung": 500, "ca_anh": 600}
PHAN = 200
THU_MUC = f"{OUT}/mo_ta_5000"
os.makedirs(THU_MUC, exist_ok=True)
tong_giay, so_anh_lam = 0.0, 0
for start in range(0, len(anh), PHAN):
    file_phan = f"{THU_MUC}/part_{start // PHAN:03d}.jsonl"
    if os.path.exists(file_phan):
        continue
    nhom = anh[start:start + PHAN]
    yeu_cau = []
    for f in nhom:
        im = Image.open(duong_dan[f]).convert("RGB")
        for k, b in enumerate(khung_all[f]["nguoi"]):
            yeu_cau.append((f, "nguoi", f"p{k}_conf{b[4]}", cat_anh(im, b[:4], pad=0.15), NGUOI))
        for ten, (mo_ta, box) in VUNG.items():
            yeu_cau.append((f, "vung", ten, cat_anh(im, box), vung_prompt(mo_ta)))
        yeu_cau.append((f, "ca_anh", "full", im, CA_ANH))
    t = time.time()
    if llm is not None:
        sps = [SamplingParams(temperature=0, max_tokens=TOI_DA[kind], repetition_penalty=1.05) for _, kind, _, _, _ in yeu_cau]
        convs = [[{"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64(img)}"}},
            {"type": "text", "text": p}]}] for _, _, _, img, p in yeu_cau]
        van_ban = [o.outputs[0].text for o in llm.chat(convs, sps, use_tqdm=False)]
    else:
        van_ban = chay_transformers([(img, p) for _, _, _, img, p in yeu_cau])
    giay = time.time() - t
    kq = {f: {"file": f, "imageID": khung_all[f]["imageID"], "nguoi": [], "vung": {}, "ca_anh": None, "loi_json": 0} for f in nhom}
    for (f, kind, key, _, _), txt in zip(yeu_cau, van_ban):
        js = doc_json(txt)
        r = kq[f]
        if js is None:
            r["loi_json"] += 1
            js = {"raw": txt[:2000]}
        if kind == "nguoi":
            r["nguoi"].append({"id": key, **js})
        elif kind == "vung":
            r["vung"][key] = js
        else:
            r["ca_anh"] = js
    tam = file_phan + ".tmp"
    with open(tam, "w", encoding="utf-8") as fh:            # ghi file tạm rồi đổi tên: không bao giờ để lại phần dở
        for f in nhom:
            fh.write(json.dumps(kq[f], ensure_ascii=False) + "\n")
    os.replace(tam, file_phan)
    tong_giay += giay; so_anh_lam += len(nhom)
    loi = sum(r["loi_json"] for r in kq.values())
    con = len(anh) - start - len(nhom)
    print(f"phần {start // PHAN:03d}: {len(nhom)} ảnh, {giay:.0f}s ({giay / len(nhom):.1f}s/ảnh), JSON lỗi {loi}/{len(yeu_cau)}"
          f" — còn {con} ảnh, ~{con * tong_giay / so_anh_lam / 3600:.1f} giờ")
    ghi_nhat_ky(f"xong phần {start // PHAN:03d} ({giay:.0f}s, JSON lỗi {loi})")
cach = "vLLM" if llm is not None else "transformers"
with open("toc_do.txt", "w", encoding="utf-8") as fh:
    fh.write(f"{tong_giay:.0f} giây cho {so_anh_lam} ảnh làm trong phiên này, {cach}, {MODEL}\n")

In [ ]:
# 7. Gộp các phần thành 1 file, kiểm tra đủ 5000 ảnh, rồi TỰ NGẮT phiên
parts = sorted(glob.glob(f"{THU_MUC}/part_*.jsonl"))
rows = [json.loads(l) for p in parts for l in open(p, encoding="utf-8") if l.strip()]
xong = {r["file"] for r in rows}
chua = [f for f in anh if f not in xong]
with open(f"{OUT}/mo_ta_5000.jsonl", "w", encoding="utf-8") as fh:
    for r in sorted(rows, key=lambda r: r["file"]):
        fh.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"{len(rows)} ảnh trong mo_ta_5000.jsonl; còn thiếu {len(chua)}; JSON lỗi {sum(r['loi_json'] for r in rows)}")
ghi_nhat_ky(f"gộp: {len(rows)} ảnh, thiếu {len(chua)}")
_hen_gio.cancel()
ngat("chạy xong" if not chua else f"còn thiếu {len(chua)} ảnh — chạy lại notebook để làm tiếp")